# 02 — Training Results & Analysis

Load saved training history and test metrics, reproduce figures for the report.

**Run after training completes** (i.e., after `python main.py`).

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.utils import plot_training_curves, get_device, load_checkpoint
from src.model import build_model
from src.dataset import get_dataloaders
from src.evaluate import evaluate, threshold_sweep

OUTPUT_DIR = '../outputs'
DATA_ROOT  = '../data/sen1floods11'   # >>>  UPDATE THIS PATH  <<<
DEVICE = get_device()

## 1. Load and plot training curves

In [ ]:
with open(os.path.join(OUTPUT_DIR, 'history.json')) as f:
    history = json.load(f)

plot_training_curves(history, save_path=os.path.join(OUTPUT_DIR, 'training_curves_nb.png'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train', color='steelblue')
axes[0].plot(epochs, history['val_loss'],   label='Val',   color='tomato')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='BCE + Dice Loss')
axes[0].legend(); axes[0].grid(alpha=0.4)

axes[1].plot(epochs, history['train_iou'], label='Train', color='steelblue')
axes[1].plot(epochs, history['val_iou'],   label='Val',   color='tomato')
axes[1].set(xlabel='Epoch', ylabel='IoU', title='Intersection over Union')
axes[1].legend(); axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.show()
print(f'Best val IoU: {max(history["val_iou"]):.4f} at epoch {history["val_iou"].index(max(history["val_iou"]))+1}')

## 2. Load test metrics

In [ ]:
with open(os.path.join(OUTPUT_DIR, 'test_metrics.json')) as f:
    metrics = json.load(f)

df = pd.DataFrame([metrics])
print('Test set metrics:')
display(df.T.rename(columns={0: 'Value'}))

## 3. Ablation comparison (if available)

In [ ]:
ablation_path = os.path.join(OUTPUT_DIR, 'ablation_results.json')
if os.path.exists(ablation_path):
    with open(ablation_path) as f:
        ablation = json.load(f)
    df_abl = pd.DataFrame(ablation).T
    print('Ablation study results:')
    display(df_abl)

    df_abl['iou'].plot(kind='bar', figsize=(8,4), color=['#4472C4','#ED7D31','#70AD47'],
                       title='IoU Comparison: Ablation Models')
    plt.ylabel('IoU'); plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()
else:
    print('No ablation results found. Run: python main.py --ablation')

## 4. View saved prediction visualisations

In [ ]:
from PIL import Image
import glob

pred_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'prediction_*.png')))[:4]

if pred_files:
    fig, axes = plt.subplots(1, len(pred_files), figsize=(6*len(pred_files), 6))
    if len(pred_files) == 1:
        axes = [axes]
    for ax, path in zip(axes, pred_files):
        ax.imshow(Image.open(path))
        ax.set_title(os.path.basename(path))
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No prediction images found in outputs/. Run evaluation first.')